# Lab 13: The Waterfall That Moves Backward

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 2, *The Time Editor*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-13.ipynb)

**What you will do:** change the strength, width and direction of neural adaptation to see how a balanced population of motion-detecting neurons ends up reporting an illusory direction of motion.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## Before you start

Staring at strong rotating or spiral motion for 30-60 seconds can cause mild dizziness in
sensitive viewers. Look away and stop if you feel uncomfortable; the aftereffect fades on its
own within seconds.

## The experiment in the book

In Lab 13 you stared at downward motion (a waterfall or an online spiral video) for 30-60
seconds, then looked at something stationary. The stationary surface appeared to drift in the
opposite direction, upward, or outward if you adapted to a contracting spiral, even though
nothing was moving. This **motion aftereffect** happens because staring at one direction of
motion fatigues the direction-selective neurons tuned to it, so when you then look at
something still, the balance between "up" and "down" detectors is skewed, and the imbalance
is read as motion.

## Record your results

Time how long the aftereffect lasts after a fixed adaptation period. Use the same online
video each time (Step 1), adapt for 30 seconds, then start a stopwatch the instant you look
away and stop it when the illusory motion disappears. Run a few trials.

In [ ]:
# example data: replace with your own stopwatch timings, in seconds
adaptation_s = 30
aftereffect_s = [8.5, 11.2, 6.9, 9.8, 12.5]  # example data: replace with yours

results = pd.DataFrame({"trial": range(1, len(aftereffect_s) + 1), "aftereffect (s)": aftereffect_s})
display(results)
print(f"Mean aftereffect duration after {adaptation_s} s of adaptation: {np.mean(aftereffect_s):.1f} s "
      f"(range {min(aftereffect_s):.1f}-{max(aftereffect_s):.1f} s)")
results["aftereffect (s)"].plot.bar(color="steelblue", figsize=(5, 3))
plt.ylabel("seconds"); plt.xlabel("trial"); plt.xticks(rotation=0); plt.show()

## Compare

Lab 13 reports that the aftereffect "typically lasts 5-15 seconds after 30 seconds of
adaptation." Since your record cell used the same 30-second adaptation period, you can
compare your mean duration directly with that range.

In [ ]:
mean_duration = np.mean(aftereffect_s)
print(f"Your mean aftereffect duration: {mean_duration:.1f} s")
if 5 <= mean_duration <= 15:
    print("Inside the typical 5-15 second range reported in Lab 13.")
elif mean_duration < 5:
    print("Shorter than typical. Possible reasons: the adaptation video did not cover enough of your"
          " visual field, you blinked or looked away during adaptation, or you are simply less"
          " susceptible to this particular illusion -- individual differences are well documented.")
else:
    print("Longer than typical. Judging exactly 'when did it stop' near the end of a fading illusion"
          " is inherently fuzzy, so some overestimate is expected; treat this as a rough estimate, not"
          " a precise measurement.")

## Simulation

After staring at a waterfall, stationary rocks appear to drift upward. Why? Neurons tuned to downward motion fatigue, shifting the balance towards upward-tuned neurons. This simulation reveals how adaptation in a population of direction-selective neurons produces an illusory signal.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

directions = np.linspace(0, 360, 360, endpoint=False)  # 360 steps, 1 degree apart
baseline = np.ones_like(directions)                     # equal activity = no motion percept

# Angle convention: standard maths convention, 0 degrees = right, 90 = up, 180 = left,
# 270 = down (counterclockwise from the positive x-axis), matching cos/sin below.
# Adaptation: neurons tuned near 270 degrees (downward) reduce their firing
adapt_centre = 270
adapt_width = 60
adaptation = 1 - 0.7 * np.exp(-0.5 * ((directions - adapt_centre) / adapt_width) ** 2)
adapted = baseline * adaptation

# Population vector: sum of all neurons' preferred directions weighted by firing rate
def pop_vector(rates, dirs_deg):
    dirs_rad = np.deg2rad(dirs_deg)
    vx = np.sum(rates * np.cos(dirs_rad))
    vy = np.sum(rates * np.sin(dirs_rad))
    angle = np.rad2deg(np.arctan2(vy, vx)) % 360
    return angle, np.sqrt(vx**2 + vy**2)

angle_after, mag_after = pop_vector(adapted, directions)
print(f"Perceived direction after adaptation: {angle_after:.0f} degrees")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(directions, baseline, "k--",
    label="Before adaptation (balanced)")
ax.fill_between(directions, adapted, alpha=0.3, color="steelblue")
ax.plot(directions, adapted, "steelblue", linewidth=2,
    label="After adaptation (fatigued at 270°)")
ax.axvline(270, color="red", linestyle=":", alpha=0.5,
    label="Adapted direction (↓ down, 270°)")
ax.axvline(angle_after, color="green", linestyle="-", linewidth=2,
    label=f"Perceived direction: {angle_after:.0f}° (↑ up)")
ax.set_xlim(-10, 370)
ax.set_xlabel("Preferred direction (°, 0 = right, 90 = up, 180 = left, 270 = down)")
ax.set_ylabel("Firing rate (normalised)")
ax.set_title("Motion Aftereffect: Adaptation Creates an Illusory Population Signal")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**What to notice:** Before adaptation, the population is perfectly balanced and has no net direction at all. The green line shows the perceived direction after adaptation—90° (upward), even though nothing is moving; the code follows the standard maths convention where 0° points right and angles increase counterclockwise, so the downward-tuned neurons that were weakened sit at 270°, directly opposite. The aftereffect is not a glitch; it is the mathematically inevitable consequence of weakening one group of neurons within an otherwise balanced population.

## Change the parameters

**depth** controls how much the adapted neurons fatigue (0 = no adaptation at all, close
to 1 = almost total fatigue). **width** controls how many neighbouring directions fatigue
along with the adapted one. **adapt_centre** is the direction that was adapted, using the
same convention as the code above (0° = right, 90° = up, 180° = left, 270° = down, so
270° is "downward", matching the waterfall). The plot and the printed angle reuse the same
`pop_vector()` function from the cell above.
(If the sliders do not appear, run the cell again.)

In [ ]:
from ipywidgets import interact, IntSlider, FloatSlider

def explore(adapt_centre=270, depth=0.7, width=60):
    adaptation = 1 - depth * np.exp(-0.5 * ((directions - adapt_centre) / width) ** 2)
    adapted = baseline * adaptation
    angle_after, mag_after = pop_vector(adapted, directions)
    opposite = (adapt_centre + 180) % 360

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(directions, baseline, "k--", label="Before adaptation (balanced)")
    ax.fill_between(directions, adapted, alpha=0.3, color="steelblue")
    ax.plot(directions, adapted, "steelblue", linewidth=2, label="After adaptation")
    ax.axvline(adapt_centre, color="red", linestyle=":", alpha=0.5, label=f"Adapted direction ({adapt_centre:.0f} degrees)")
    ax.axvline(angle_after, color="green", linewidth=2, label=f"Perceived direction: {angle_after:.0f} degrees")
    ax.set_xlabel("Preferred direction (degrees)")
    ax.set_ylabel("Firing rate (normalised)")
    ax.set_title(f"depth={depth}, width={width}")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    print(f"Adapted direction: {adapt_centre:.0f} degrees  |  exact opposite: {opposite:.0f} degrees")
    print(f"Perceived (illusory) direction: {angle_after:.0f} degrees  |  population-vector strength: {mag_after:.2f}")

interact(explore,
         adapt_centre=IntSlider(value=270, min=0, max=350, step=10),
         depth=FloatSlider(value=0.7, min=0.0, max=0.95, step=0.05),
         width=IntSlider(value=60, min=10, max=120, step=5));

## The AI side

The lab's AI Connection compares the motion aftereffect to **concept drift** and
**calibration drift** in engineered systems: a sensor or model that is well-calibrated at
first can drift out of calibration after sustained, one-sided input, and needs periodic
**bias correction** to reset it. Below, a simulated sensor gradually drifts after being fed
a one-sided signal, the same way a population of motion detectors adapts after one-sided
motion, and a basic recalibration step corrects it, the way your own neurons recover once
the adapting input stops.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
n_steps = 100
one_sided_input = np.ones(60)          # 60 steps of one-sided ("downward") input
no_input = np.zeros(n_steps - 60)      # then it stops, like looking away

true_signal = np.concatenate([one_sided_input, no_input])
bias = 0.0
drift_rate = 0.01     # how fast the sensor's zero point drifts under sustained one-sided input
recovery_rate = 0.05  # how fast it recalibrates once the input stops

biases, readings = [], []
for t, signal in enumerate(true_signal):
    if signal > 0:
        bias += drift_rate * (1 - bias)   # sustained input drifts the baseline
    else:
        bias -= recovery_rate * bias      # no input: baseline recovers towards zero
    reading = signal - bias + rng.normal(0, 0.02)   # what the sensor now reports
    biases.append(bias); readings.append(reading)

df = pd.DataFrame({"true signal": true_signal, "sensor bias": biases, "sensor reading": readings})
display(df.iloc[[0, 30, 59, 60, 65, 80, 99]])

plt.figure(figsize=(7, 3.5))
plt.plot(df["true signal"], "k--", label="true input")
plt.plot(df["sensor bias"], color="firebrick", label="accumulated bias (drift)")
plt.plot(df["sensor reading"], color="steelblue", alpha=0.7, label="sensor reading")
plt.axvline(60, color="grey", linestyle=":", label="input stops (looking away)")
plt.xlabel("time step"); plt.legend(fontsize=8); plt.show()

Notice that right after the input stops (step 60), the sensor still reports a negative
reading even though the true signal is zero, a false signal caused entirely by the
accumulated bias, just as the stationary surface looks like it is moving even though
nothing is. The bias then decays back towards zero, mirroring your own recovery. The
disanalogy: this toy sensor's drift and recovery rates are hand-set constants; real
direction-selective neurons adapt and recover through actual biophysical processes
(synaptic depression, ion channel dynamics) that this simulation only caricatures.

## Pool the class data

Pool the class's aftereffect durations, and relate them to how long each person adapted for
(Lab 13 states that "longer adaptation produces longer aftereffects"). Each row is one
anonymous ID, their adaptation time in seconds, and their aftereffect duration in seconds.

In [ ]:
import io
csv_text = """id,adaptation_s,aftereffect_s
P01,15,4.8
P01,30,9.1
P01,60,15.2
P02,15,3.9
P02,30,7.5
P02,60,13.8
P03,15,5.5
P03,30,10.4
P03,60,17.1
P04,15,4.1
P04,30,8.7
P04,60,14.5
"""  # example data: replace with your class CSV, e.g. pd.read_csv("mae_class.csv")
class_df = pd.read_csv(io.StringIO(csv_text))
display(class_df.groupby("adaptation_s")["aftereffect_s"].agg(["mean", "std", "count"]))

r = np.corrcoef(class_df["adaptation_s"], class_df["aftereffect_s"])[0, 1]
boot_r = [np.corrcoef(*class_df.sample(len(class_df), replace=True)[["adaptation_s", "aftereffect_s"]].values.T)[0, 1]
          for _ in range(2000)]
lo, hi = np.percentile(boot_r, [2.5, 97.5])
print(f"Correlation between adaptation time and aftereffect duration: r = {r:.2f} "
      f"(95% bootstrap interval {lo:.2f}-{hi:.2f})")

class_df.plot.scatter(x="adaptation_s", y="aftereffect_s", color="steelblue", figsize=(5, 4))
plt.xlabel("adaptation time (s)"); plt.ylabel("aftereffect duration (s)"); plt.show()

## Questions to think about

1. Set depth to 0. What happens to the perceived direction, and why does this match "no illusion" when nothing has adapted?
2. Raise depth toward its maximum of 0.95. Does the perceived direction move much, or does mostly the population-vector strength change? What would a larger depth correspond to experientially -- a shorter or a longer stare at the waterfall?
3. Change adapt_centre to 0, so the adapted neurons were tuned to rightward motion instead of downward. Where does the illusory perceived direction end up, and does it match the "opposite direction" prediction described in the lab's Brain Mechanism section?
4. Increase width so a broader range of neighbouring directions fatigues together. Does the perceived direction change, or mainly the shape and strength of the dip in the firing-rate curve? Relate this to Lab 07's point that adaptation happens in populations of neurons, not single cells.

## Challenge

The lab notes that "longer adaptation produces longer aftereffects" and that this scaling
tells you something about the underlying mechanism. The `depth` parameter above is a rough
stand-in for how much a population has fatigued after a given staring duration. Use the
loop below, which reuses `directions`, `baseline` and `pop_vector()` from the simulation
above, to see what the model predicts about the relationship between fatigue and the
strength of the illusory signal.

In [ ]:
depths = np.linspace(0, 0.95, 20)
magnitudes = []
for d in depths:
    adaptation = 1 - d * np.exp(-0.5 * ((directions - 270) / 60) ** 2)
    adapted = baseline * adaptation
    _, mag = pop_vector(adapted, directions)
    magnitudes.append(mag)

plt.figure(figsize=(6, 3.5))
plt.plot(depths, magnitudes, "o-", color="steelblue")
plt.xlabel("Adaptation depth (a proxy for staring duration)")
plt.ylabel("Population-vector magnitude (a proxy for aftereffect strength)")
plt.title("Does more fatigue predict a stronger illusion?")
plt.show()
print("This shows what the population-vector mechanism implies, not a replication of")
print("Barlow and Hill's (1963) original timing data -- treat it as a model prediction to test.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-13.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")